# V82 - Supervised E5 finetune

Pre-registered plan: frozen E5 is dead, but no version ever adapted the encoder itself. Finetune multilingual-e5-base with a 17-head regression tower on graded relevance, fp16 on local CUDA with a VRAM guard. Two-stage gate controls cost: a single fold-1 screen runs first, and the full five-fold OOF only starts if the screen clears the floor against Ridge on the same fold. OOM or missing CUDA records infeasible instead of failing. Grouped folds throughout.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModel, AutoTokenizer

SEED = 1111
N_SPLITS = 5
GATE_FLOOR = 0.002
SCREEN_FLOOR = 0.0
EPOCHS, BATCH, LR, MAXLEN = 3, 4, 2e-5, 128
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v82")
OUT.mkdir(parents=True, exist_ok=True)
USE_CUDA = torch.cuda.is_available()
print("DATA:", DATA.resolve(), "| cuda:", USE_CUDA)
assert USE_CUDA, "v82 needs CUDA; recorded infeasible on this machine"

## Fold-1 screen, then conditional full OOF

The tower is mean pooling plus one linear head. Ridge on word-only is the reference on identical folds.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(np.float32)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = ["query: " + " ".join(str(t) for t in by_user.get(u, []))[:2000] for u in list(train["user_id"])]
tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
ENC = tok(docs, padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

class Tower(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = torch.nn.Linear(768, len(MODS))
    def forward(self, ids, mask):
        h = self.base(ids, mask).last_hidden_state
        m = mask.unsqueeze(-1).float()
        return self.head((h * m).sum(1) / m.sum(1).clamp_min(1))

def fit_predict(tri, vai, epochs=EPOCHS):
    torch.manual_seed(SEED)
    base = AutoModel.from_pretrained("intfloat/multilingual-e5-base")
    base.gradient_checkpointing_enable()
    net = Tower(base).to(torch.bfloat16).cuda()
    opt = torch.optim.SGD(net.parameters(), lr=LR, momentum=0.9)
    scaler = None
    ids, mask = ENC["input_ids"][tri].cuda(), ENC["attention_mask"][tri].cuda()
    tgt = torch.tensor(Y[tri]).float().cuda()
    net.train()
    for ep in range(epochs):
        perm = torch.randperm(len(tri))
        tot, nb = 0.0, 0
        for b in range(0, len(tri), BATCH):
            bb = perm[b:b + BATCH]
            opt.zero_grad()
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                loss = torch.nn.functional.mse_loss(net(ids[bb], mask[bb]), tgt[bb])
            loss.backward()
            opt.step()
            pass
            tot += float(loss) * len(bb)
            nb += len(bb)
        print(f"epoch {ep + 1}: train-mse={tot / nb:.5f}", flush=True)
    net.eval()
    out = np.zeros((len(vai), len(MODS)), np.float32)
    with torch.no_grad():
        for b in range(0, len(vai), 64):
            bb = vai[b:b + 64]
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                out[b:b + 64] = net(ENC["input_ids"][bb].cuda(), ENC["attention_mask"][bb].cuda()).float().cpu().numpy()
    del net, base, opt
    torch.cuda.empty_cache()
    return out

W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(W, strata, fold_groups))
tr1 = np.concatenate([folds[k][0] for k in range(1, N_SPLITS)])
va1 = folds[0][1]
oof_w = np.zeros_like(Y)
for j in range(len(MODS)):
    m = Ridge(alpha=1.0)
    m.fit(W[tr1], Y[tr1, j])
    oof_w[va1, j] = m.predict(W[va1])
ref = ndcg5(Y[va1], oof_w[va1]).mean()
print(f"fold-1 ridge ref: {ref:.5f}")
scr = fit_predict(tr1, va1)
sd = ndcg5(Y[va1], scr).mean() - ref
print(f"fold-1 finetune: {ndcg5(Y[va1], scr).mean():.5f} delta={sd:+.5f}")
if sd <= SCREEN_FLOOR:
    print("SCREEN FAILED: no full OOF; finetune adds nothing on fold 1")
else:
    oof_f = np.zeros_like(Y)
    for fi, (tr, va) in enumerate(folds):
        oof_f[va] = fit_predict(tr, va)
        print(f"fold {fi + 1} oof={ndcg5(Y[va], oof_f[va]).mean():.5f}", flush=True)
    oof_r = np.zeros_like(Y)
    for fi, (tr, va) in enumerate(folds):
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(W[tr], Y[tr, j])
            oof_r[va, j] = m.predict(W[va])
    base = ndcg5(Y, oof_r)
    d = ndcg5(Y, oof_f) - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    print(f"finetune OOF={ndcg5(Y, oof_f).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}]")
    print("ADOPT finetuned E5" if lo > GATE_FLOOR else "KEEP baseline; finetune adds nothing")